# UK Biobank — Academic Impact: the main-paper panel

**Status: placeholder.** The panel is not built yet. This notebook exists so that the
selection is made in *one* place, on the record, rather than by whoever assembles the
figure last.

Analysis 03 currently draws **14 figures** across two notebooks. The main paper gets one
multi-panel figure. This notebook is where the shortlist is chosen and assembled.

| source notebook | figures | what they cover |
|---|---|---|
| [`03_academic_impact_01_for_analysis.ipynb`](03_academic_impact_01_for_analysis.ipynb) | 11 | volume, activity index, citation impact, quality bands, growth — by **field** |
| [`03_academic_impact_02_citation.ipynb`](03_academic_impact_02_citation.ipynb) | 3 | entry cohorts, impact portfolios, influence fingerprint — by **author** |

## How to fill this in

1. Run §1. It inventories what is actually on disk under the 03 slug — do not select from
   memory or from this table, which will drift.
2. Put the chosen stems in `PANEL` in §2, with the letter each becomes in the figure.
3. §3 assembles them.

## The one decision that has to be made first

There are two ways to build a panel, and they are not interchangeable:

| | **re-draw** (recommended) | **paste** |
|---|---|---|
| how | import the chart code, draw into a shared `GridSpec` | place the saved PDFs/PNGs side by side |
| type sizes | one scale across the panel, set by `STYLE` | each sub-figure keeps its own; they will not match |
| axes | can be shared, aligned, deduplicated | whatever each figure already had |
| cost | the chart cells have to be callable — they are currently inline in their notebooks | none |
| revision | re-runs from data | has to be regenerated upstream first |

**Re-drawing is the right answer for a main-paper figure**, and it has a prerequisite this
notebook cannot skip: the chart cells chosen in §2 must first be lifted out of their
notebooks into functions that take an `ax`. That is the real work, and it should be done
only for the three or four charts that make the cut — not for all 14.

Until the selection exists, §3 refuses to guess.


## 1. What is on disk

Inventoried, not remembered. Every figure analysis 03 has written, with its size and age,
so the shortlist is made against the current run rather than against a stale list.


In [1]:
import sys
from pathlib import Path

# Anchor every path on the repo root regardless of where the notebook is launched from
# (repo root, src/, or src/data_analysis/). See shared_paths for the why.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()

import pandas as pd

# Same style section as the two source notebooks, so a panel drawn here inherits the
# palette and type sizes its panels were drawn with (D4: keyed on the exact section name).
from utils.shared_style import load_style, savefig
STYLE = load_style("03_academic_impact")

FIG_DIR = P.FIG_ACADEMIC_IMPACT
TABLE_DIR = P.OUTPUT_TABLES / "03_academic_impact"

# One row per figure STEM, collapsing the pdf/svg/png triplet each savefig writes.
rows = {}
for f in sorted(FIG_DIR.glob("*.*")):
    if f.suffix.lower() not in (".pdf", ".svg", ".png"):
        continue
    r = rows.setdefault(f.stem, {"stem": f.stem, "formats": [], "kb": 0,
                                 "modified": pd.Timestamp(f.stat().st_mtime, unit="s")})
    r["formats"].append(f.suffix.lstrip("."))
    r["kb"] = max(r["kb"], f.stat().st_size // 1024)

inventory = pd.DataFrame(rows.values())
if inventory.empty:
    print(f"NOTHING IN {FIG_DIR}.\nRun the two source notebooks first — neither writes "
          f"unless STYLE['save'] is true.")
else:
    inventory["formats"] = inventory.formats.apply(lambda v: ",".join(sorted(v)))
    inventory = inventory.sort_values("stem").reset_index(drop=True)
    print(f"{len(inventory)} figure(s) under {FIG_DIR.relative_to(ROOT)}:\n")
    display(inventory)
    print(f"\n{len(list(TABLE_DIR.glob('*.csv')))} table(s) under "
          f"{TABLE_DIR.relative_to(ROOT)} — the numbers behind them.")

# HOW MANY *SHOULD* BE HERE. An inventory that quietly lists a third of the candidates is
# worse than no inventory: the shortlist gets made from whatever happened to be on disk.
# The field notebook saves its charts under a leading chart NUMBER ("04_activity_index_…"),
# the author notebook under a "fig…" stem, so the two arms are told apart by their names.
EXPECTED = {"field": 11, "author": 3}          # savefig() calls in each source notebook
on_disk = {"field": sum(s[0].isdigit() for s in rows),
           "author": sum(not s[0].isdigit() for s in rows)}

for arm, want in EXPECTED.items():
    got = on_disk[arm]
    if got < want:
        print(f"\n!! {arm.upper()} ARM: {got} of {want} figures on disk.")
        if arm == "field":
            print("   The field notebook inherits `save: false` from style.base in "
                  "universal_settings.yml,\n   so its charts are drawn inline and never "
                  "written. Before selecting, open\n   "
                  "03_academic_impact_01_for_analysis.ipynb, add `STYLE[\"save\"] = True` "
                  "after load_style,\n   and re-run it — otherwise 11 of the 14 candidates "
                  "cannot be seen from here.")


3 figure(s) under output/figures/data_analysis/03_academic_impact:



,stem,formats,kb,modified
0,fig3a_author_entry_cohort_academic_impact_profile,"pdf,png,svg",267,2026-08-26 18:27:39.420999765
1,fig_author_impact_portfolio_map,"pdf,png,svg",7732,2026-08-26 18:27:41.254817485
2,fig_author_influence_fingerprint_heatmap,"pdf,png,svg",679,2026-08-26 18:27:44.072130919



11 table(s) under output/tables/03_academic_impact — the numbers behind them.

!! FIELD ARM: 0 of 11 figures on disk.
   The field notebook inherits `save: false` from style.base in universal_settings.yml,
   so its charts are drawn inline and never written. Before selecting, open
   03_academic_impact_01_for_analysis.ipynb, add `STYLE["save"] = True` after load_style,
   and re-run it — otherwise 11 of the 14 candidates cannot be seen from here.


## 2. The selection — **to be filled in**

One row per panel, in reading order. `stem` must match a `stem` from §1 exactly.

`caption` is the one line that will sit under the panel letter in the figure legend; write
it here rather than in the paper draft, so the figure and its caption move together.

Leave `PANEL` empty until the shortlist is agreed — §3 checks it and stops rather than
inventing one.


In [2]:
# panel letter -> (figure stem, caption)
#
# Fill this in. Example of the intended shape, with the two strongest candidates from
# each arm — COMMENTED OUT because the selection has not been made:
#
# PANEL = {
#     "a": ("04_activity_index_for_L4",
#           "What UK Biobank is disproportionately about"),
#     "b": ("05_impact_map_for_L4_mncs",
#           "Where it concentrates, and where that work lands"),
#     "c": ("09_footprint_quality_median_for_L4",
#           "Its share of each field, cut at the field's own median and top decile"),
#     "d": ("fig3a_author_entry_cohort_academic_impact_profile",
#           "Who produced it: contribution by author-entry cohort"),
# }

PANEL: dict[str, tuple[str, str]] = {}

# A main-paper figure is one page. Beyond about four panels the type has to shrink below
# what a journal will accept, so the ceiling is a constraint rather than a preference.
MAX_PANELS = 4


## 3. Assemble

Checks the selection against what §1 found, then stops — deliberately.

The assembly step is not written because it cannot be written until §2 names the charts:
each chosen chart has to be lifted from its notebook into a function taking an `ax`, and
which functions those are *is* the selection. See the table at the top for why re-drawing
beats pasting for this particular figure.


In [3]:
if not PANEL:
    print("No selection yet — see §2.\n")
    if not inventory.empty:
        print("Candidates from §1, for the shortlist:\n")
        for stem in inventory.stem:
            print(f"    {stem}")
else:
    known = set(inventory.stem)
    missing = {k: v[0] for k, v in PANEL.items() if v[0] not in known}
    if missing:
        raise KeyError(
            f"these panel stems are not in {FIG_DIR.name}: {missing}. "
            f"A stem must match §1 exactly — re-run the source notebook if it is absent.")
    if len(PANEL) > MAX_PANELS:
        raise ValueError(f"{len(PANEL)} panels selected; the page holds {MAX_PANELS}.")

    print(f"{len(PANEL)} panel(s) selected:\n")
    for letter, (stem, caption) in PANEL.items():
        print(f"  ({letter})  {stem}\n       {caption}")

    # Record the selection beside the tables, so the paper draft and this notebook cannot
    # drift apart about which figure is which panel.
    pd.DataFrame(
        [{"panel": k, "stem": v[0], "caption": v[1]} for k, v in PANEL.items()]
    ).to_csv(TABLE_DIR / "main_panel_selection.csv", index=False)
    print(f"\nselection written to {TABLE_DIR / 'main_panel_selection.csv'}")

    raise NotImplementedError(
        "Selection is valid; assembly is not written yet. Next step: lift the chart cell "
        "for each stem above out of its notebook into a draw_<name>(ax, ...) function in "
        "src/utils/, then call them into a GridSpec here.")


No selection yet — see §2.

Candidates from §1, for the shortlist:

    fig3a_author_entry_cohort_academic_impact_profile
    fig_author_impact_portfolio_map
    fig_author_influence_fingerprint_heatmap
